# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imalik-7/Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

My chosen lane is Refresh / Content Opportunity Scoring. I frame it primarily as a ranking/scoring problem because the decision is not simply whether a page is good or bad. The useful question is: which pages should a content reviewer inspect first? The system should assign each eligible page a priority score and rank the pages from highest to lowest review priority. The output supports actions such as refresh, expand, review CTR, or monitor.## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# Find the starter dataset.
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists() and "google.colab" in sys.modules:
    REPO_DIR = Path("/content/flyrank-ml-internship-starter")

    if not REPO_DIR.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                str(REPO_DIR),
            ],
            check=True,
        )

    os.chdir(REPO_DIR)
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

# Same basic eligibility used by the starter workflow.
lane_df = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

print("Lane: Refresh / Content Opportunity Scoring")
print("ML task type: Ranking / Scoring")
print("Rows available:", len(lane_df))

Lane: Refresh / Content Opportunity Scoring
ML task type: Ranking / Scoring
Rows available: 30000


For this starter exercise, I will use current decline as a temporary proxy for whether a page may deserve review. I define the proxy as trend_direction == "down".

This is not an ideal final target because trend_direction is itself calculated from current-window performance rather than from a future observed outcome. I will therefore treat it honestly as a starter proxy, not as ground truth.

A stronger future version of this project would use information available before a decision point and measure an observed outcome afterward, for example:

previous 90 days of signals → decline during the following 30 days.

That would give the project a genuinely future-looking target while avoiding information from the outcome window leaking into the features.

In [2]:
# Temporary starter proxy.
lane_df["is_declining_proxy"] = (
    lane_df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

proxy_count = lane_df["is_declining_proxy"].sum()
proxy_rate = lane_df["is_declining_proxy"].mean()

print(f"Total pages: {len(lane_df):,}")
print(f"Declining-proxy pages: {proxy_count:,}")
print(f"Declining-proxy rate: {proxy_rate:.1%}")

lane_df[
    ["content_id", "trend_direction", "is_declining_proxy"]
].head()

Total pages: 30,000
Declining-proxy pages: 16,262
Declining-proxy rate: 54.2%


,content_id,trend_direction,is_declining_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1


My primary success metric is Precision@50.

This matches the real decision because a content team usually cannot review every page at once. If reviewers can inspect 50 pages, Precision@50 asks: of the 50 pages ranked highest by the system, how many match the review proxy?

I will consider a learned ranking useful if it performs better than the transparent starter rule baseline on held-out data. The committed starter baseline has a Precision@50 of 0.240, so my minimum definition of improvement is a Precision@50 above 0.24 under honest validation.

I prefer this metric over generic accuracy because the practical goal is to make the top of the review queue useful, not simply classify every page correctly.

In [3]:
K = 50
STARTER_BASELINE_P50 = 0.240

def precision_at_k(y_true, scores, k=50):
    """
    Fraction of positive target rows among the k highest-scored rows.
    """
    result = pd.DataFrame({
        "target": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = result.nlargest(k, "score")
    return top_k["target"].mean()

print("Primary success metric: Precision@50")
print("Review capacity (K):", K)
print(f"Starter rule baseline Precision@50: {STARTER_BASELINE_P50:.3f}")
print(
    "Minimum success condition:",
    f"beat {STARTER_BASELINE_P50:.3f} on honest held-out validation"
)

Primary success metric: Precision@50
Review capacity (K): 50
Starter rule baseline Precision@50: 0.240
Minimum success condition: beat 0.240 on honest held-out validation


The unit of analysis is one content page/item. After applying the starter eligibility conditions and deduplicating by content_id, each row represents one pseudonymized content item that could potentially be placed in the review queue.

The page has observable signals such as impressions, clicks, sessions, age, freshness, CTR, average position, engagement, and word count. The IDs are used only for identifying, grouping, or validation; they should not be treated as predictive features.

In [4]:
display_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "trend_direction",
    "is_declining_proxy",
]

unit_df = lane_df[display_columns].copy()

print("Unit of analysis: one eligible content item/page")
print("Rows:", len(unit_df))
print("Unique content IDs:", unit_df["content_id"].nunique())
print("Duplicate content IDs:", unit_df["content_id"].duplicated().sum())

unit_df.head(10)

Unit of analysis: one eligible content item/page
Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,avg_position,ctr,engagement_rate,scroll_rate,word_count,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,29,17,187,20,10.6,0.76,5.88,4.55,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,445,25,20.3,0.05,0.00,10.00,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,141,20,36.5,0.09,0.00,28.57,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,78,463,22,6.2,0.49,1.28,3.45,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,263,14,44.0,0.13,0.00,24.29,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,147,20,8.5,0.03,0.00,25.00,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,0,1,90,20,7.0,0.00,0.00,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,28,445,22,21.2,0.06,3.57,7.14,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,68,90,20,46.0,0.09,5.88,6.25,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,3,257,104,4.9,0.16,0.00,0.00,NaN,down,1


A fixed rule is a useful baseline, but the prioritization problem contains several signals that can interact in different ways. A page can be old but still perform well, have high impressions but weak CTR, have good rankings but weak engagement, or show decline despite strong existing demand.

A hand-written rule has to decide fixed thresholds and weights for all of these situations. ML may improve the ranking because it can learn combinations and interactions among visibility, age, position, CTR, engagement, content depth, and other observable signals rather than applying the same fixed thresholds to every page.

ML only earns its place if it beats a transparent fixed-rule baseline under honest validation. If it does not, I should keep the simpler rule.

The output is still decision support: a high-ranked page means review this page earlier, not “automatically rewrite this page” or “a refresh will definitely improve performance.”

In [ ]:
signal_flags = pd.DataFrame(index=lane_df.index)

signal_flags["stale"] = (
    lane_df["days_since_last_update"] >= 180
)

signal_flags["high_visibility"] = (
    lane_df["impressions_90d"] >= 500
)

signal_flags["strong_position"] = (
    (lane_df["avg_position"] > 0) &
    (lane_df["avg_position"] <= 20)
)

signal_flags["low_ctr"] = (
    (lane_df["ctr"] > 0) &
    (lane_df["ctr"] < 0.5)
)

signal_flags["low_engagement"] = (
    ((lane_df["engagement_rate"] > 0) &
     (lane_df["engagement_rate"] < 30))
    |
    ((lane_df["scroll_rate"] > 0) &
     (lane_df["scroll_rate"] < 30))
)

signal_flags["thin_content"] = (
    (lane_df["word_count"] > 0) &
    (lane_df["word_count"] < 1200)
)

print("Pages triggering each signal:")
print(signal_flags.sum().sort_values(ascending=False))

# Count how many different combinations of these signals appear.
patterns = signal_flags.astype(int).astype(str).agg("".join, axis=1)

print("\nDifferent signal combinations in the data:", patterns.nunique())
print(
    "Pages triggering 2 or more signals:",
    (signal_flags.sum(axis=1) >= 2).sum()
)

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.